# Laporan Capstone: Pipeline Pengolahan Data untuk Pencocokan Profil CV dan Lowongan Pekerjaan

Dokumen ini menyajikan alur kerja komprehensif mulai dari tahap akuisisi data, standarisasi teks, hingga pembentukan dataset akhir yang siap digunakan dalam proses pelatihan model pencocokan otomatis.

## 1. Pengolahan Dataset Job Description (JD)

### 1.1 Library

In [1]:
import re
import json
import ast
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

### 1.2 Load Dataset

In [2]:
file_jd = '13_tLGL1b2Drk6F-ep9Xc3_wTzrzydrJh'
url_jd = f'https://drive.google.com/uc?id={file_jd}'

df = pd.read_csv(url_jd)
print('Jumlah data awal:', df.shape)
display(df.head())

Jumlah data awal: (3000, 9)


,ID,Query,Job Title,Description,IT Skills,Soft Skills,Education,Experience,Token Usage
0,3859,Artificial Intelligence,Collaborative Manipulation Roboticist,"Location: Schlumberger-Doll Research, Cambridg...","Collaborative Manipulation, Artificial Intelli...","Problem-solving, Organizational skills, Commun...",NaN,These skills are related to experience.,562
1,3764,Artificial Intelligence,Software Engineer - Innovation Lab,About DENSO DENSO is one of the largest global...,"Software Development, Infotainment Systems, Au...","Creativity, Collaboration, Problem-solving, Ad...",NaN,NaN,616
2,3597,Artificial Intelligence,"Director, Standards & Strategy","As a Director of Strategy & Strategy at Xperi,...","Audio/video codec, Media streaming and storage...","Participation in international standards, cons...","Skills Related to Education:, Undergraduate de...","Skills Related to Experience:, Audio/video cod...",549
3,3746,Artificial Intelligence,Business Strategy Consultant,If you have a strategic mindset and expertise ...,"Mobility technological trends expertise, Busin...","Strategic mindset, Thought leadership, Communi...",NaN,NaN,613
4,3872,Artificial Intelligence,Regular Full-Time,Mission Who We Are Founded and continuously le...,"Test planning, Bug tracking tools, Automation ...","Team player, Entrepreneurial mindset, Problem-...",NaN,NaN,1051


### 1.3 Penyesuaian Struktur Kolom
Langkah ini bertujuan untuk menyeragamkan nama kolom dan menghapus informasi yang tidak relevan dengan kebutuhan pemodelan.

In [3]:
df = df.rename(columns={
    'ID'         : 'job_id',
    'Query'      : 'industry_domain',
    'Job Title'  : 'job_title',
    'Description': 'job_desc_text',
    'IT Skills'  : 'required_skills',
    'Soft Skills': 'soft_skills',
})

df = df.drop(columns=['Education', 'Experience', 'Token Usage'])
print('Daftar kolom setelah penyesuaian:', df.columns.tolist())

Daftar kolom setelah penyesuaian: ['job_id', 'industry_domain', 'job_title', 'job_desc_text', 'required_skills', 'soft_skills']


### 1.4 Fungsi Pembersihan Teks
Fungsi ini digunakan untuk menghapus tag HTML, URL, dan karakter khusus yang dapat mengganggu performa model.

In [4]:
def clean_text_universal(text):
    if pd.isna(text):
        return ""
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

### 1.5 Persiapan Basis Data Model
Menyiapkan dataset yang telah dibersihkan untuk tahap pemrosesan lanjut tanpa menyertakan kolom yang tidak diperlukan untuk pemodelan.

In [5]:
df_model = df[['job_id', 'industry_domain', 'job_title', 'job_desc_text', 'required_skills']].copy()
df_model = df_model.dropna(subset=['job_desc_text', 'required_skills']).reset_index(drop=True)

print(f'Basis data model siap: {df_model.shape[0]} baris.')

Basis data model siap: 2990 baris.


### 1.6 Tahap Pra-proses Lanjutan
Pada tahap ini, dilakukan proses deduplikasi data, penghapusan teks yang terlalu pendek, serta pembersihan noise pada daftar skill agar data lebih berkualitas.

In [6]:
df_model = df[['job_id', 'industry_domain', 'job_title', 'job_desc_text', 'required_skills']].copy()
df_model = df_model.dropna(subset=['job_desc_text', 'required_skills']).reset_index(drop=True)

# Deduplikasi dan standarisasi teks
df_model = df_model.drop_duplicates(subset=['job_desc_text']).reset_index(drop=True)
df_model['job_desc_text'] = df_model['job_desc_text'].apply(clean_text_universal)

def clean_required_skills(skills_str):
    if pd.isna(skills_str): return ""
    noise_patterns = ['note:', 'technical skills:', 'required']
    items = [s.strip().lower() for s in str(skills_str).split(',')]
    cleaned = [i for i in items if i and not any(p in i for p in noise_patterns)]
    return ', '.join(cleaned)

df_model['required_skills'] = df_model['required_skills'].apply(clean_required_skills)

jd_train, jd_test = train_test_split(
    df_model, test_size=0.2, random_state=42, stratify=df_model['industry_domain']
)

print(f'Dataset JD siap: {len(jd_train)} train, {len(jd_test)} test.')

Dataset JD siap: 2330 train, 583 test.


In [7]:
id_dibuang = [5921, 8664, 6015, 7005, 5555]
jd_train = jd_train[~jd_train['job_id'].isin(id_dibuang)]

### 1.7 Ekspor Data JD
Data disimpan ke dalam format CSV untuk memisahkan data latih (train) dan data uji (test).

In [25]:
import os
OUTPUT_DIR = './output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Simpan file intermediate ke folder output
jd_train.to_csv(os.path.join(OUTPUT_DIR, 'jd_train.csv'), index=False)
jd_test.to_csv(os.path.join(OUTPUT_DIR, 'jd_test.csv'),   index=False)

print(f'Penyimpanan file dataset JD ke {OUTPUT_DIR} selesai.')

Penyimpanan file dataset JD ke ./output selesai.


## 2. Pengolahan Dataset Curriculum Vitae (CV)

### 2.1 Library & Load Dataset

In [9]:
from datasets import load_dataset

ds1 = load_dataset("InferencePrince555/Resume-Dataset")
df1 = pd.DataFrame(ds1['train'])

it_categories = [
    'Generate a Resume for a Database Administrator Job',
    'Generate a Resume for a Java Developer Job',
    'Generate a Resume for a Network Administrator Job',
    'Generate a Resume for a Python Developer Job',
    'Generate a Resume for a Security Analyst Job',
    'Generate a Resume for a Software Developer Job',
    'Generate a Resume for a Systems Administrator Job',
    'Generate a Resume for a Web Developer Job',
]

df_it = df1[df1['instruction'].isin(it_categories)]
print('Data mentah CV berhasil dimuat. Total baris:', len(df_it))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

updated_data_final_cleaned.csv:   0%|          | 0.00/206M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/32481 [00:00<?, ? examples/s]

Data mentah CV berhasil dimuat. Total baris: 25640


### 2.2 Kategorisasi Role Kandidat
Menerapkan pemetaan kategori agar role kandidat sesuai dengan struktur domain industri IT yang ditargetkan.

In [10]:
role_mapping = {
    'Generate a Resume for a Database Administrator Job'   : 'Database Administrator',
    'Generate a Resume for a Java Developer Job'           : 'Java Developer',
    'Generate a Resume for a Network Administrator Job'    : 'Network Administrator',
    'Generate a Resume for a Python Developer Job'         : 'Python Developer',
    'Generate a Resume for a Security Analyst Job'         : 'Security Analyst',
    'Generate a Resume for a Software Developer Job'       : 'Software Developer',
    'Generate a Resume for a Systems Administrator Job'    : 'Systems Administrator',
    'Generate a Resume for a Web Developer Job'            : 'Web Developer',
}

df_it = df_it.rename(columns={'Resume_test': 'cv_text'})
df_it['candidate_role'] = df_it['instruction'].map(role_mapping)
df_it = df_it.drop(columns=['instruction', 'input'])
df_it = df_it[['candidate_role', 'cv_text']].copy()

print('Total CV IT (sebelum cleaning & sampling):', len(df_it))
print('Per kategori:')
print(df_it['candidate_role'].value_counts().sort_index())

Total CV IT (sebelum cleaning & sampling): 25640
Per kategori:
candidate_role
Database Administrator    2784
Java Developer            2502
Network Administrator     2260
Python Developer          2359
Security Analyst          2259
Software Developer        5828
Systems Administrator     4182
Web Developer             3466
Name: count, dtype: int64


### 2.3 Standarisasi Profil dan Ekstraksi Kompetensi
Proses ini menggunakan fungsi pembersihan universal untuk menyeragamkan format teks CV. Selanjutnya, dilakukan ekstraksi keahlian menggunakan daftar kompetensi IT terkurasi untuk menghasilkan fitur pendukung dalam pencocokan data.

In [11]:
# Menggunakan fungsi universal yang sudah didefinisikan di bagian 1.4
IT_SKILLS_LIST = [
    'python', 'java', 'javascript', 'typescript', 'c++', 'c#', 'c', 'sql', 'mysql',
    'postgresql', 'mongodb', 'oracle', 'react', 'angular', 'vue', 'nodejs', 'django',
    'flask', 'fastapi', 'spring boot', 'aws', 'azure', 'gcp', 'docker', 'kubernetes',
    'machine learning', 'deep learning', 'nlp', 'data science', 'tableau', 'power bi'
]

IT_SKILLS_LIST_DEDUP = sorted(list(set(IT_SKILLS_LIST)), key=lambda x: (-len(x.split()), x))

def extract_skills(text):
    if pd.isna(text): return ''
    text_lower = text.lower()
    found = []
    for skill in IT_SKILLS_LIST_DEDUP:
        pattern = r'(?<![a-zA-Z0-9])' + re.escape(skill) + r'(?![a-zA-Z0-9])'
        if re.search(pattern, text_lower):
            found.append(skill)
    return ', '.join(found)

print(f'Proses ekstraksi siap dengan {len(IT_SKILLS_LIST_DEDUP)} entitas skill.')

Proses ekstraksi siap dengan 31 entitas skill.


### 2.4 Sampling dan Validasi Dataset
Melakukan penyeimbangan jumlah data antar kategori untuk menghindari bias model, serta memastikan tidak ada data yang bocor antara set latih dan uji.

In [12]:
SAMPLE_PER_CATEGORY = 2250

# Pembersihan dan sampling data CV
df_it['cv_text'] = df_it['cv_text'].apply(clean_text_universal)
df_it = df_it.drop_duplicates(subset=['cv_text']).reset_index(drop=True)

df_cv = df_it.groupby('candidate_role').apply(lambda x: x.sample(n=min(len(x), SAMPLE_PER_CATEGORY), random_state=42)).reset_index(drop=True)
df_cv['candidate_skills'] = df_cv['cv_text'].apply(extract_skills)

# Validasi akhir ketersediaan data skill
df_cv = df_cv[df_cv['candidate_skills'] != ''].reset_index(drop=True)
df_cv.insert(0, 'cv_id', range(1, len(df_cv) + 1))

cv_train, cv_test = train_test_split(df_cv, test_size=0.2, random_state=42, stratify=df_cv['candidate_role'])
print(f'Dataset CV siap: {len(cv_train)} train, {len(cv_test)} test.')

/tmp/ipykernel_952/2066604946.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cv = df_it.groupby('candidate_role').apply(lambda x: x.sample(n=min(len(x), SAMPLE_PER_CATEGORY), random_state=42)).reset_index(drop=True)


Dataset CV siap: 10957 train, 2740 test.


### 2.5 Simpan File CV

In [13]:
# Simpan file intermediate ke folder output
cv_train.to_csv(os.path.join(OUTPUT_DIR, 'cv_train.csv'), index=False)
cv_test.to_csv(os.path.join(OUTPUT_DIR, 'cv_test.csv'),   index=False)

print(f'Dataset CV telah berhasil diekspor ke {OUTPUT_DIR}.')

Dataset CV telah berhasil diekspor ke ./output.


## 3. Integrasi Data dan Persiapan Dataset Akhir

Tahap ini melibatkan penggabungan dataset CV dan Job Description untuk membentuk pasangan data yang akan digunakan pada model komparasi serta pemberian label kategori untuk identifikasi entitas.

### 3.1 Definisi Relasi Peran dan Domain
Menentukan hubungan antara kategori pekerjaan pada CV dengan domain industri pada Job Description untuk memastikan validitas pasangan data.

In [14]:
SEMANTIC_MAP = {
    "Database Administrator": [
        "Database Administrator",
        "Data Architect",
        "Data Engineer",
        "Data Quality Manager",
        "Data Warehousing",
    ],
    "Java Developer": [
        "Full Stack Developer",
        "Cloud Services Developer",
        "Technology Integration",
        "Cloud Architect",
    ],
    "Network Administrator": [
        "Network Architect",
        "IT Systems Administrator",
        "Technical Operations",
    ],
    "Python Developer": [
        "Data Scientist",
        "Machine Learning",
        "Deep Learning",
        "Artificial Intelligence",
        "Data Engineer",
        "Full Stack Developer",
    ],
    "Security Analyst": [
        "Information Security Analyst",
        "IT Systems Administrator",
        "IT Consultant",
    ],
    "Software Developer": [
        "Full Stack Developer",
        "Cloud Services Developer",
        "Technology Integration",
        "Cloud Architect",
    ],
    "Systems Administrator": [
        "IT Systems Administrator",
        "Technical Operations",
        "Cloud Architect",
    ],
    "Web Developer": [
        "Full Stack Developer",
        "Data Visualization Expert",
        "Technology Integration",
    ],
}

mapping_rows = [
    {'candidate_role': role, 'industry_domain': domain, 'is_positive_match': 1}
    for role, domains in SEMANTIC_MAP.items()
    for domain in domains
]
pd.DataFrame(mapping_rows).to_csv('semantic_mapping_table.csv', index=False)
print(f'Tabel pemetaan semantik berhasil dibuat dengan {len(mapping_rows)} entri.')

Tabel pemetaan semantik berhasil dibuat dengan 31 entri.


### 3.2 Konstruksi Dataset Berpasangan (Matching Pairs)
Proses ini menghasilkan pasangan data antara kandidat dan lowongan pekerjaan dengan label kecocokan (1) atau ketidakcocokan (0). Kriteria seleksi didasarkan pada ambang batas kemiripan keahlian teknis (Skill Overlap).

In [26]:
import numpy as np
import pandas as pd

TARGET_POSITIVE_MATCHES = 350
MIN_SKILL_OVERLAP = 5

def generate_curated_dataset(cv_df, jd_df, semantic_map):
    print("Menghasilkan dataset pasangan CV-JD terkurasi...")
    final_pairs = []
    num_pos = 0

    for _, cv in cv_df.sample(n=len(cv_df), random_state=42).iterrows():
        if num_pos >= TARGET_POSITIVE_MATCHES: break
        pos_domains = semantic_map.get(cv['candidate_role'], [])
        pos_pool = jd_df[jd_df['industry_domain'].isin(pos_domains)]
        cv_s = set([s.strip().lower() for s in str(cv['candidate_skills']).split(',') if s.strip()])

        for _, jd in pos_pool.sample(n=min(len(pos_pool), 300)).iterrows():
            jd_s = set([s.strip().lower() for s in str(jd['required_skills']).split(',') if s.strip()])
            if len(cv_s.intersection(jd_s)) >= MIN_SKILL_OVERLAP:
                final_pairs.append({
                    'cv_text': cv['cv_text'][:2000], 'job_desc_text': jd['job_desc_text'][:2000],
                    'match_label': 1, 'cv_skills': cv['candidate_skills'], 'jd_skills': jd['required_skills']
                })
                num_pos += 1
                break

    num_neg = 0
    for _, cv in cv_df.sample(n=len(cv_df), random_state=123).iterrows():
        if num_neg >= num_pos: break
        related = semantic_map.get(cv['candidate_role'], [])
        neg_pool = jd_df[~jd_df['industry_domain'].isin(related)]

        for _, jd in neg_pool.sample(n=min(len(neg_pool), 100)).iterrows():
            cv_words = set(str(cv['cv_text']).lower().split())
            jd_words = set(str(jd['job_desc_text']).lower().split())
            overlap_ratio = len(cv_words & jd_words) / len(cv_words | jd_words)

            if overlap_ratio < 0.01:
                final_pairs.append({
                    'cv_text': cv['cv_text'][:2000], 'job_desc_text': jd['job_desc_text'][:2000],
                    'match_label': 0, 'cv_skills': cv['candidate_skills'], 'jd_skills': jd['required_skills']
                })
                num_neg += 1
                break

    return pd.DataFrame(final_pairs)

siamese_curated = generate_curated_dataset(cv_train, jd_train, SEMANTIC_MAP)
# save file
siamese_curated.to_csv('siamese_curated_data_v7.csv', index=False)
print("Dataset terkurasi v7 selesai dibuat.")

Menghasilkan dataset pasangan CV-JD terkurasi...
Dataset terkurasi v7 selesai dibuat.


### 3.3 Evaluasi Kualitas Dataset Pasangan
Melakukan pengecekan terhadap distribusi label dan tingkat kemiripan antar pasangan untuk memastikan kualitas data sebelum masuk ke tahap pelatihan.

In [16]:
import pandas as pd
import os

# Muat data terkurasi v7 untuk verifikasi
file_path = 'siamese_curated_data_v7.csv'
if os.path.exists(file_path):
    train_final = pd.read_csv(file_path)
    print(f"SIAP UNTUK PEMODELAN:")
    print(f"- Nama File: {file_path}")
    print(f"- Total Baris: {len(train_final)}")
    print(f"- Pasangan Positif: {len(train_final[train_final['match_label']==1])}")
    print(f"- Pasangan Negatif: {len(train_final[train_final['match_label']==0])}")
    display(train_final.head())
else:
    print(f"⚠️ File {file_path} belum dibuat.")

SIAP UNTUK PEMODELAN:
- Nama File: siamese_curated_data_v7.csv
- Total Baris: 700
- Pasangan Positif: 350
- Pasangan Negatif: 350


,cv_text,job_desc_text,match_label,cv_skills,jd_skills
0,web ui developer web ui span ldeveloperspan we...,"control-tec, an aptiv company, is a global pro...",1,"angular, java, javascript, mongodb, mysql, nod...","java, python, mysql, mongodb, message queues, ..."
1,front endui developer front endui span ldevelo...,are you a passionate developer with a desire t...,1,"angular, c, java, javascript, mongodb, mysql, ...","net, c, java, javascript/typescript, html5, cs..."
2,javaj2ee developer span ljavaspanj2ee span lde...,rally health is all about putting health in th...,1,"spring boot, angular, aws, c, docker, java, ja...","****, scala, play, akka, mongodb, postgresql, ..."
3,python developer span lpythonspan span ldevelo...,about us foxtrot is the next generation corner...,1,"angular, aws, c, django, docker, flask, java, ...","software engineering, systems design, systems ..."
4,sr javaj2ee developer sr span ljavaspanj2ee sp...,"as a engineer, you will get to be part of a te...",1,"spring boot, angular, aws, java, javascript, m...","python, react, postgresql, flask, aws, html, c..."


### 3.2.1 Status Kesiapan Dataset
Dataset pasangan (matching pairs) telah berhasil dikonstruksi dengan label seimbang untuk kebutuhan pelatihan model Siamese Network.

In [17]:
# Menampilkan statistik singkat label untuk verifikasi akhir
print("Distribusi Label Matching:")
print(siamese_curated['match_label'].value_counts())
print("\nContoh data siap proses:")
display(siamese_curated[['match_label', 'cv_skills', 'jd_skills']].head())

Distribusi Label Matching:
match_label
1    350
0    350
Name: count, dtype: int64

Contoh data siap proses:


,match_label,cv_skills,jd_skills
0,1,"angular, java, javascript, mongodb, mysql, nod...","java, python, mysql, mongodb, message queues, ..."
1,1,"angular, c, java, javascript, mongodb, mysql, ...","net, c, java, javascript/typescript, html5, cs..."
2,1,"spring boot, angular, aws, c, docker, java, ja...","****, scala, play, akka, mongodb, postgresql, ..."
3,1,"angular, aws, c, django, docker, flask, java, ...","software engineering, systems design, systems ..."
4,1,"spring boot, angular, aws, java, javascript, m...","python, react, postgresql, flask, aws, html, c..."


---

### 3.3 Penyelarasan Format NER (BIO Tagging)
Langkah ini bertujuan untuk mentransformasi teks mentah menjadi format token yang terlabeli (BIO), sehingga model dapat mengenali entitas keahlian teknis secara spesifik dari dalam kalimat.

In [18]:
GLOBAL_SKILL_SET = set(IT_SKILLS_LIST_DEDUP)

def parse_skills_cv(skills_str):
    if pd.isna(skills_str):
        return []
    skills = [s.strip().lower() for s in str(skills_str).split(',') if s.strip()]
    return sorted(skills, key=lambda x: (-len(x.split()), x))

def parse_skills_jd(skills_str, global_skills_set):
    per_row = set()
    if not pd.isna(skills_str):
        for s in str(skills_str).split(','):
            s = s.strip().lower()
            if s:
                per_row.add(s)
    combined = list(per_row | global_skills_set)
    return sorted(combined, key=lambda x: (-len(x.split()), x))

def bio_tag(text, skills_list):
    if pd.isna(text) or not skills_list:
        return []

    tokens = str(text).lower().split()
    labels = ['O'] * len(tokens)

    for skill in skills_list:
        skill_tokens = skill.split()
        n = len(skill_tokens)
        for i in range(len(tokens) - n + 1):
            if tokens[i:i+n] == skill_tokens:
                if all(labels[i + j] == 'O' for j in range(n)):
                    labels[i] = 'B-SKILL'
                    for j in range(1, n):
                        labels[i + j] = 'I-SKILL'

    return list(zip(tokens, labels))

def skill_coverage(bio_list):
    if not bio_list:
        return 0.0
    return round(sum(1 for _, l in bio_list if l != 'O') / len(bio_list), 4)

def apply_bio_tagging_cv(df):
    df = df.copy()
    df['bio_tags'] = df.apply(
        lambda row: bio_tag(row['cv_text'], parse_skills_cv(row['candidate_skills'])),
        axis=1,
    )
    df['bio_skill_coverage'] = df['bio_tags'].apply(skill_coverage)
    return df

def apply_bio_tagging_jd(df):
    df = df.copy()
    skills_combined = df['required_skills'].apply(
        lambda x: parse_skills_jd(x, GLOBAL_SKILL_SET)
    )
    df['bio_tags'] = df.apply(
        lambda row: bio_tag(row['job_desc_text'], skills_combined[row.name]),
        axis=1,
    )
    df['bio_skill_coverage'] = df['bio_tags'].apply(skill_coverage)
    return df

cv_train_bio = apply_bio_tagging_cv(cv_train)
cv_test_bio  = apply_bio_tagging_cv(cv_test)
jd_train_bio = apply_bio_tagging_jd(jd_train)
jd_test_bio  = apply_bio_tagging_jd(jd_test)

print('Hasil BIO Tagging:')
for name, df in [('cv_train', cv_train_bio), ('cv_test',  cv_test_bio),
                 ('jd_train', jd_train_bio), ('jd_test',  jd_test_bio)]:
    zero = (df['bio_skill_coverage'] == 0).sum()
    print(f'  {name:10s} — avg coverage: {df["bio_skill_coverage"].mean():.2%} '
          f'| zero coverage: {zero} baris ({zero/len(df)*100:.1f}%)')

sample = cv_train_bio.iloc[0]
print(f'\nContoh BIO CV (role={sample["candidate_role"]}):')
print(f'  Skills : {sample["candidate_skills"][:80]}')
print(f'  15 token pertama:')
for tok, lbl in sample['bio_tags'][:15]:
    print(f'    {tok:22s}  {lbl}{"  ← SKILL" if lbl != "O" else ""}')

sample_jd = jd_train_bio.iloc[0]
print(f'\nContoh BIO JD (domain={sample_jd["industry_domain"]}):')
print(f'  Skills : {str(sample_jd["required_skills"])[:80]}')
print(f'  15 token pertama:')
for tok, lbl in sample_jd['bio_tags'][:15]:
    print(f'    {tok:22s}  {lbl}{"  ← SKILL" if lbl != "O" else ""}')

Hasil BIO Tagging:
  cv_train   — avg coverage: 2.56% | zero coverage: 0 baris (0.0%)
  cv_test    — avg coverage: 2.52% | zero coverage: 1 baris (0.0%)
  jd_train   — avg coverage: 2.75% | zero coverage: 109 baris (4.7%)
  jd_test    — avg coverage: 2.84% | zero coverage: 26 baris (4.5%)

Contoh BIO CV (role=Network Administrator):
  Skills : data science, java, oracle
  15 token pertama:
    business                O
    analyst                 O
    business                O
    analyst                 O
    erie                    O
    pa                      O
    work                    O
    experience              O
    business                O
    analyst                 O
    tata                    O
    consultancy             O
    services                O
    gurgaon                 O
    haryana                 O

Contoh BIO JD (domain=Data Engineer):
  Skills : data engineering, etl tools (informatica, talend), sql database queries, java pr
  15 token pertama:
    da

In [19]:
import json

def serialize_bio(df, bio_col='bio_tags'):
    df = df.copy()
    df[bio_col] = df[bio_col].apply(json.dumps)
    return df

ner_cv_cols = ['cv_id', 'candidate_role', 'candidate_skills', 'bio_tags', 'bio_skill_coverage']
ner_jd_cols = ['job_id', 'industry_domain', 'required_skills',  'bio_tags', 'bio_skill_coverage']

# File NER mentah (sebelum filtrasi) masuk ke output
serialize_bio(cv_train_bio[ner_cv_cols]).to_csv(os.path.join(OUTPUT_DIR, 'cv_ner_train.csv'), index=False)
serialize_bio(cv_test_bio[ner_cv_cols]).to_csv(os.path.join(OUTPUT_DIR, 'cv_ner_test.csv'),   index=False)
serialize_bio(jd_train_bio[ner_jd_cols]).to_csv(os.path.join(OUTPUT_DIR, 'jd_ner_train.csv'), index=False)
serialize_bio(jd_test_bio[ner_jd_cols]).to_csv(os.path.join(OUTPUT_DIR, 'jd_ner_test.csv'),   index=False)

print(f'File NER mentah berhasil disimpan di {OUTPUT_DIR}.')

File NER mentah berhasil disimpan di ./output.


---

### 3.4 Optimasi Distribusi Label & Pipeline NER
Melakukan penyaringan terhadap label 'O' (non-entity) yang berlebihan untuk menjaga keseimbangan dataset selama proses pelatihan model NER. Langkah ini menggabungkan pembersihan tag dan perbaikan format tokenisasi menjadi satu alur kerja otomatis untuk menghasilkan dataset final yang optimal.

In [20]:
try:
    from nltk.corpus import stopwords
    import nltk
    nltk.download('stopwords', quiet=True)
    STOP_WORDS = set(stopwords.words('english'))
except Exception:
    # Fallback jika nltk module gagal dimuat
    STOP_WORDS = set(['i', 'me', 'my', 'we', 'you', 'is', 'are', 'the', 'and', 'at'])

def fast_ner_pipeline(input_name, output_name):
    input_path = os.path.join(OUTPUT_DIR, f'{input_name}.csv')
    if not os.path.exists(input_path):
        return

    df = pd.read_csv(input_path)
    df['bio_tags'] = df['bio_tags'].apply(ast.literal_eval)

    tokens_list, tags_list = [], []
    for bio_list in df['bio_tags']:
        t_rev, g_rev = [], []
        for tok, tag in bio_list:
            if tag != 'O':
                t_rev.append(tok)
                g_rev.append(tag)
            elif tok.lower() not in STOP_WORDS and len(tok) > 1 and tok.isalnum():
                t_rev.append(tok)
                g_rev.append(tag)
        tokens_list.append(t_rev)
        tags_list.append(g_rev)

    new_df = pd.DataFrame({
        'id': df['cv_id'] if 'cv_id' in df.columns else df['job_id'],
        'tokens': tokens_list,
        'tags': tags_list
    })

    # Hapus baris dengan skill < 3
    mask = new_df['tags'].apply(lambda x: sum(1 for t in x if t != 'O')) >= 3
    new_df_filtered = new_df[mask].copy()

    new_df_filtered.to_csv(f'{output_name}.csv', index=False)
    print(f'Exported: {output_name}.csv')

# Eksekusi pipeline data latih dan uji
fast_ner_pipeline('cv_ner_train', 'cv_ner_train_final')
fast_ner_pipeline('jd_ner_train', 'jd_ner_train_final')
fast_ner_pipeline('cv_ner_test', 'cv_ner_test')
fast_ner_pipeline('jd_ner_test', 'jd_ner_test')

Exported: cv_ner_train_final.csv
Exported: jd_ner_train_final.csv
Exported: cv_ner_test.csv
Exported: jd_ner_test.csv


### 3.5 Verifikasi Entitas NER
Langkah verifikasi ini memastikan bahwa proses transformasi BIO tagging telah menghasilkan token-token yang valid dan siap digunakan oleh model NER.

In [21]:
# Verifikasi sederhana format data
for file in ['cv_ner_train_final.csv', 'jd_ner_train_final.csv']:
    if os.path.exists(file):
        df_check = pd.read_csv(file)
        print(f"{file}: {len(df_check)} baris terverifikasi.")

cv_ner_train_final.csv: 8997 baris terverifikasi.
jd_ner_train_final.csv: 2047 baris terverifikasi.


**Status Akhir:** Dataset telah divalidasi dan siap untuk tahap pemodelan.

---

## 4. Finalisasi dan Ekspor Akhir
Pengumpulan seluruh dataset yang telah diproses ke dalam paket akhir untuk kebutuhan pengiriman data.

In [22]:
CREATE_ZIP_HANDOVER = True # @param {type:"boolean"}

if CREATE_ZIP_HANDOVER:
    import shutil
    import os
    from google.colab import files

    zip_name = "capstone_dataset"
    final_csvs = [
        'siamese_curated_data_v7.csv',
        'semantic_mapping_table.csv',
        'cv_ner_train_final.csv',
        'jd_ner_train_final.csv',
        'cv_ner_test.csv',
        'jd_ner_test.csv'
    ]

    temp_pkg = 'handover_package'
    os.makedirs(temp_pkg, exist_ok=True)

    # Verifikasi file sebelum dipacking
    missing_files = []
    for f in final_csvs:
        if os.path.exists(f):
            shutil.copy(f, os.path.join(temp_pkg, f))
            print(f"✅ {f} siap dipacking.")
        else:
            missing_files.append(f)

    if missing_files:
        print(f"⚠️ PERINGATAN! File berikut ter-skip karena tidak ditemukan: {missing_files}")

    if len(os.listdir(temp_pkg)) > 0:
        shutil.make_archive(zip_name, 'zip', temp_pkg)
        shutil.rmtree(temp_pkg)
        files.download(f'{zip_name}.zip')
        print(f"\nDownload dimulai: {zip_name}.zip")
    else:
        print("❌ Gagal membuat ZIP: Tidak ada file yang ditemukan.")
else:
    print("Creation skipped.")

✅ siamese_curated_data_v7.csv siap dipacking.
✅ semantic_mapping_table.csv siap dipacking.
✅ cv_ner_train_final.csv siap dipacking.
✅ jd_ner_train_final.csv siap dipacking.
✅ cv_ner_test.csv siap dipacking.
✅ jd_ner_test.csv siap dipacking.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download dimulai: capstone_dataset.zip


### Ekspor Dataset ke Google Drive
Bagian ini akan menyinkronkan hasil dataset final ke folder Google Drive Anda agar dapat diakses oleh anggota AI.

In [24]:
UPLOAD_TO_DRIVE = True # @param {type:"boolean"}

if UPLOAD_TO_DRIVE:
    from google.colab import drive
    import os
    import shutil

    try:
        drive.mount('/content/drive', force_remount=True)
        drive_path = '/content/drive/My Drive/Capstone_Datasets'
        os.makedirs(drive_path, exist_ok=True)

        sync_files = [
            'siamese_curated_data_v7.csv',
            'semantic_mapping_table.csv',
            'cv_ner_train_final.csv',
            'jd_ner_train_final.csv',
            'cv_ner_test.csv',
            'jd_ner_test.csv'
        ]

        print(f"Memulai sinkronisasi ke: {drive_path}")
        count = 0
        for f in sync_files:
            if os.path.exists(f):
                shutil.copy(f, os.path.join(drive_path, f))
                print(f"✅ Berhasil upload: {f}")
                count += 1
            else:
                print(f"❌ ERROR: File {f} tidak ditemukan di workspace!")

        print(f"\nTotal: {count}/{len(sync_files)} file berhasil disinkronkan.")
    except Exception as e:
        print(f"Sync error: {e}")

Mounted at /content/drive
Memulai sinkronisasi ke: /content/drive/My Drive/Capstone_Datasets
✅ Berhasil upload: siamese_curated_data_v7.csv
✅ Berhasil upload: semantic_mapping_table.csv
✅ Berhasil upload: cv_ner_train_final.csv
✅ Berhasil upload: jd_ner_train_final.csv
✅ Berhasil upload: cv_ner_test.csv
✅ Berhasil upload: jd_ner_test.csv

Total: 6/6 file berhasil disinkronkan.


---
## Data Dictionary

### 1. Siamese Matching Dataset (`siamese_curated_data_v7.csv`)

| Kolom | Tipe Data | Deskripsi |
|---|---|---|
| `cv_text` | String | Cuplikan teks profil kandidat (max 2000 chars) |
| `job_desc_text` | String | Cuplikan teks deskripsi pekerjaan (max 2000 chars) |
| `match_label` | Integer | Label biner: **1** (Cocok/Match), **0** (Tidak Cocok/No Match) |
| `cv_skills` | String | Daftar skill yang diekstrak dari CV |
| `jd_skills` | String | Daftar skill yang diekstrak dari JD |

### 2. NER Dataset (`cv_ner_train_final.csv`, `jd_ner_train_final.csv`)

| Kolom | Tipe Data | Deskripsi |
|---|---|---|
| `id` | Integer | ID unik referensi (CV ID atau Job ID) |
| `tokens` | List | Kumpulan kata (tokens) hasil pembersihan |
| `tags` | List | Label BIO tagging: `B-SKILL` (awal), `I-SKILL` (lanjutan), `O` (luar entitas) |

### 3. Semantic Reference (`semantic_mapping_table.csv`)

| Kolom | Tipe Data | Deskripsi |
|---|---|---|
| `candidate_role` | String | Nama kategori peran kandidat |
| `industry_domain` | String | Nama kategori domain industri pada lowongan |
| `is_positive_match` | Integer | Status validitas relasi (Default: 1) |